# Conversation dynamics in podcasts: An analysis of interruptions, calls to action and sentiment across genres
This project aims to analyze conversational dynamics in podcasts using the SPoRC dataset, which provides annotated podcast transcriptions with speaker and dialogue turn information. In the project, we focus on three key elements of conversational dynamics; interruptions, calls to action, and sentiment. These will be the lenses for understanding how podcast conversations differ across genres. By analyzing the existence and nature of interruptions, the use of persuasive and action oriented language toward the listeners, along with the sentiments and emotional tone, we wish to compare and characterize how conversational patterns vary between genres of podcasts. The analysis aims at sheding light on how podcasts communicate and engage audiences differently depending on their genre.

This notebook consists of an initial data exploration pipeline, and then methodological considerations for each of the parts of the analysis, which will be conducted in the P3 milestone part of this project.

## Data exploration
In this part of the notebook we have worked with loading and understanding the data to ensure the feasibility of our ideas on the actual dataset. We have calculated the descriptive statistics of the dataset at both podcast, episode and turn levels. Furthermore, we have plotted the content of the dataset and done initial interaction with the data to understand the available columns in each part of the dataset. We have used the sporc library to interact with the data, and familiarized ourselves with the available functionality in here. 

### Import dependencies

In [2]:
from helpers.sporc import (
    SPORCDataset,
    get_all_categories,
    get_main_categories,
    get_subcategories_list,
    is_main_category,
    is_subcategory,
    is_valid_category,
)
from pprint import pprint

/Users/nicolinesorensen/opt/anaconda3/envs/nlp/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


### Load data

I have appropriate the `sporc` library to our needs. You can load the full data from hugging face (if logged in) using 
        
    sporc = SPORCDataset()

It is also possible to use `streaming` so you wont load any data before iterating over it
    
    sporc = SPORCDataset(streaming=True)

If you have the compressed files (.gz) locally they can be read as

    sporc = SPORCDataset(local_data_dir="data")

However, since the full data is really large we will only be using the provided sample, which can be loaded as follows

In [3]:
sporc = SPORCDataset(local_data_dir="data", load_samples_only=True)

INFO:helpers.sporc.dataset:Loading SPORC dataset from local directory: data
INFO:helpers.sporc.dataset:Loading SPORC sample dataset only.
INFO:helpers.sporc.dataset:✓ All required files found
INFO:helpers.sporc.dataset:Loading all records from local files into memory...
INFO:helpers.sporc.dataset:Loading episode_data_sample...
INFO:helpers.sporc.dataset:Loading speaker_turn_data_sample...
INFO:helpers.sporc.dataset:✓ Loaded 210,000 total records from 2 files
INFO:helpers.sporc.dataset:✓ Local dataset loaded successfully in 4.98 seconds
INFO:helpers.sporc.dataset:✓ Dataset loaded successfully with 210000 total records
INFO:helpers.sporc.dataset:Processing dataset into Podcast and Episode objects...
INFO:helpers.sporc.dataset:Separating episode data from speaker turn data...


Loading episodes:   0%|          | 0/210000 [00:00<?, ?it/s]

INFO:helpers.sporc.dataset:✓ Separation completed in 0.28 seconds
INFO:helpers.sporc.dataset:  Episode records: 10,000, Speaker turn records: 200,000
INFO:helpers.sporc.dataset:Grouping episodes by podcast...
INFO:helpers.sporc.dataset:✓ Grouping completed in 0.03 seconds
INFO:helpers.sporc.dataset:  Episodes grouped into 1826 podcasts
INFO:helpers.sporc.dataset:Creating Podcast and Episode objects...
INFO:helpers.sporc.dataset:✓ Object creation completed in 0.69 seconds
INFO:helpers.sporc.dataset:Loading speaker turn data for episodes...
INFO:helpers.sporc.dataset:Loading turn data for 200,000 speaker turn records...
INFO:helpers.sporc.dataset:Grouping turns by episode URL...
INFO:helpers.sporc.dataset:✓ Turn grouping completed in 0.23 seconds
INFO:helpers.sporc.dataset:  Turns grouped into 1,033 episodes
INFO:helpers.sporc.dataset:Loading turns for each episode...
INFO:helpers.sporc.dataset:✓ Turn loading completed in 1.21 seconds
INFO:helpers.sporc.dataset:  Episodes with turns: 1,0

In [5]:
stats = sporc.get_dataset_statistics()

print(f"Total podcasts: {stats['total_podcasts']}")
print(f"Total episodes: {stats['total_episodes']}")
print(f"Total duration: {stats['total_duration_hours']:.1f} hours")
print(f"Average episode length: {stats['avg_episode_duration_minutes']:.1f} minutes")

print("\nTop categories:")
for category, count in sorted(stats['category_distribution'].items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {category}: {count} episodes")
    
print("\nTop Languages:")
for category, count in sorted(stats['language_distribution'].items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {category}: {count} episodes")
    
print("\nTop Episode Types:")
for category, count in sorted(stats['episode_types'].items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {category}: {count} episodes")

Total podcasts: 1826
Total episodes: 10000
Total duration: 5760.4 hours
Average episode length: 34.6 minutes

Top categories:
  : 73612 episodes
  religion: 2109 episodes
  spirituality: 2106 episodes
  society: 1609 episodes
  culture: 1609 episodes

Top Languages:
  en: 7926 episodes
  en-us: 1261 episodes
  en-US: 482 episodes
  en-gb: 163 episodes
  en-au: 39 episodes

Top Episode Types:
  long_form: 4838 episodes
  short_form: 2059 episodes
  solo: 1836 episodes
  interview: 1613 episodes
  panel: 684 episodes


In [7]:
# Demonstrate category utility functions
print("Category Utility Functions:")

# Get all categories
all_categories = get_all_categories()
main_categories = get_main_categories()
subcategories = get_subcategories_list()

print(f"   Total categories: {len(all_categories)}")
print(f"   Main categories: {len(main_categories)}")
print(f"   Subcategories: {len(subcategories)}")

# Show some main categories
print(f"   Sample main categories: {main_categories[:5]}")
print(f"   Sample subcategories: {subcategories[:5]}")
print()

# Demonstrate category validation
print("Category Validation:")
test_categories = ["Education", "Astronomy", "Invalid Category", "Language Learning"]

for category in test_categories:
    is_valid = is_valid_category(category)
    is_main = is_main_category(category)
    is_sub = is_subcategory(category)

    print(f"   '{category}':")
    print(f"     Valid: {is_valid}")
    print(f"     Main category: {is_main}")
    print(f"     Subcategory: {is_sub}")

print()

Category Utility Functions:
   Total categories: 110
   Main categories: 19
   Subcategories: 91
   Sample main categories: ['Arts', 'Business', 'Comedy', 'Education', 'Fiction']
   Sample subcategories: ['After Shows', 'Alternative Health', 'Animation & Manga', 'Astronomy', 'Automotive']

Category Validation:
   'Education':
     Valid: True
     Main category: True
     Subcategory: False
   'Astronomy':
     Valid: True
     Main category: False
     Subcategory: True
   'Invalid Category':
     Valid: False
     Main category: False
     Subcategory: False
   'Language Learning':
     Valid: True
     Main category: False
     Subcategory: True



In [ ]:
# Demonstrate category hierarchy & distribution 
# Note that an episode can have multiple categories
sporc.plot_category_sunburst() 

Note: This plot is interactive and not visible in the rendered notebook in Github. Instead see this 

![image](figures/sunburst.png?raw=true)

### Interacting with the dataset

Through `sporc` the data can be accessed through `Podcast` and `Episode` objects

In [22]:
# Accessing podcast and episode data
podcasts = sporc.get_all_podcasts() # List of Podcast objects
episodes = sporc.get_all_episodes() # List of Episode objects

print(f"Total podcasts: {len(podcasts)}")
print("Example podcast:", podcasts[0])

print(f"\nTotal episodes: {len(episodes)}")  
print("Example episode:", episodes[0])

Total podcasts: 1826
Example podcast: Podcast('SingOut SpeakOut', 5 episodes, 0.7h)

Total episodes: 10000
Example episode: Episode('Best of SingOut SpeakOut No.3', 13.4min, 1 speakers)


In [23]:
for i, podcast in enumerate(podcasts[:5]):
    print(f"[{i+1}] {podcast.title} ({len(podcast.episodes)} episodes)")

[1] SingOut SpeakOut (5 episodes)
[2] Brazen Education (2 episodes)
[3] California Community Colleges Podcast (1 episodes)
[4] Puck Bunnies (1 episodes)
[5] Anxiety 2 Confidence podcast (6 episodes)


#### Working with Podcasts

In [24]:
# searching for a podcast
podcast = sporc.search_podcast("singout speakout")

print(f"Title: {podcast.title}")
pprint(f"Description: {podcast.description}")
print(f"Number of episodes: {podcast.num_episodes}")
print(f"Total duration: {podcast.total_duration_hours:.1f} hours")
print(f"Hosts: {podcast.host_names}")
print(f"Categories: {podcast.categories}")

Title: SingOut SpeakOut
('Description: Award winning Australian singer-songwriter Simon Shapiro '
 'delivers his songs, writing and philosophy in a unique combination of idea '
 'sharing, including conversation, live performance and recorded music.')
Number of episodes: 5
Total duration: 0.7 hours
Hosts: ['Simon Shapiro']
Categories: ['', 'culture', 'education', 'music', 'philosophy', 'self improvement', 'society']


In [78]:
pprint(podcast.get_episode_statistics())

{'avg_episode_duration_minutes': 8.546666666666667,
 'category_distribution': {'': 20,
                           'culture': 5,
                           'education': 5,
                           'music': 5,
                           'philosophy': 5,
                           'self improvement': 5,
                           'society': 5},
 'date_range': {'earliest': '2020-05-17T12:00:00',
                'latest': '2020-06-20T23:00:00'},
 'episode_types': {'interview': 1,
                   'long_form': 0,
                   'panel': 0,
                   'short_form': 4,
                   'solo': 4},
 'guest_names': ['Lee Walker'],
 'host_names': ['Simon Shapiro'],
 'max_episode_duration_minutes': 13.383333333333333,
 'median_episode_duration_minutes': 7.933333333333334,
 'min_episode_duration_minutes': 6.0,
 'num_episodes': 5,
 'speaker_distribution': {1: 1, 3: 3, 4: 1},
 'total_duration_hours': 0.7122222222222222}


In [25]:
# Iterate through episodes
for episode in podcast.episodes:
    print(f"Episode: {episode.title}")
    print(f"Duration: {episode.duration_minutes:.1f} minutes")
    print(f"Date: {episode.episode_date}")
    print("---")

Episode: Quarterlife Crisis
Duration: 8.5 minutes
Date: 2020-05-17 12:00:00
---
Episode: Saturn Return
Duration: 7.9 minutes
Date: 2020-05-24 11:00:00
---
Episode: Today Is Yesterday
Duration: 6.9 minutes
Date: 2020-05-31 13:00:00
---
Episode: It's All Gone
Duration: 6.0 minutes
Date: 2020-06-14 14:00:00
---
Episode: Best of SingOut SpeakOut No.3
Duration: 13.4 minutes
Date: 2020-06-20 23:00:00
---


#### Working with Episodes

In [26]:
episode = podcast.episodes[0]

print(f"Title: {episode.title}")
pprint(f"Description: {episode.description}")
print(f"Duration: {episode.duration_minutes:.1f} minutes")
print(f"Hosts: {episode.host_names}")
print(f"Guests: {episode.guest_names}")
print(f"Categories: {episode.categories}")
print(f"Main speakers: {episode.num_main_speakers}")
pprint(f"Transcript: {episode.transcript[:300]}...")


Title: Quarterlife Crisis
('Description: <p>Big news week. The band Simon lived in the USA with, KisTone '
 'has released their debut album after 12 years of red tape. Simon runs '
 'through a brief history of the band and then introduces track 1, Quarterlife '
 'Crisis.</p>')
Duration: 8.5 minutes
Hosts: ['Simon Shapiro']
Guests: ['Lee Walker']
Categories: ['music', 'society', 'culture', 'philosophy', 'education', 'self improvement', '', '', '', '']
Main speakers: 3
("Transcript: I'm Simon Shapiro and this is Sing Out Speak Out. Here I give "
 "you the philosophies, ideas and songs I've been working on for many years "
 "but until now I've shared just too few of because I've been afraid of you "
 "and worse I've been afraid of me. My will to connect with you and share the "
 'best of me has...')


In [ ]:
# Episode type flags
print(f"Is solo episode: {episode.is_solo}")
print(f"Is interview: {episode.is_interview}")
print(f"Is panel discussion: {episode.is_panel}")
print(f"Is long-form: {episode.is_long_form}")
print(f"Has guests: {episode.has_guests}")

Is solo episode: False
Is interview: True
Is panel discussion: False
Is long-form: False
Has guests: True


In [27]:
pprint(episode.get_turn_statistics())

{'avg_turn_duration': 33.97866666666667,
 'avg_words_per_turn': 47.0,
 'role_distribution': {'NO_INFERRED_ROLE': 12, 'host': 3},
 'speaker_distribution': {'SPEAKER_00': 6,
                          'SPEAKER_01': 5,
                          'SPEAKER_02': 4,
                          'SPEAKER_03': 4},
 'total_turns': 15,
 'total_words': 705}


In [29]:
# Filter by runtime
long_episodes = sporc.search_episodes(min_duration=1800) # episodes longer than 30 min
short_episodes = sporc.search_episodes(max_duration=600) # episodes shorter than 10 min

# Filter by speaker participation
solo_episodes = sporc.search_episodes(max_speakers=1)
group_episodes = sporc.search_episodes(min_speakers=3)

# Filter by host identity
episodes_by_simon = sporc.search_episodes(host_name="Simon Shapiro")

# Filter by thematic category
sport_episodes = sporc.search_episodes(category="sports")
music_episodes = sporc.search_episodes(category="music")

# Combine filters for more specific search
specefic_episodes = sporc.search_episodes(
    category="education", 
    min_speakers=2,
    min_duration=1800  
)

# Sampling examples
# Randomly sample 100 sports-related episodes
sampled_sport_episodes = sporc.search_episodes(
    category="sports",
    sampling_mode="random",
    max_episodes=100
)
# Select the first 50 long-form episodes
first_long_episodes = sporc.search_episodes(
    min_duration=1800,
    sampling_mode="first",
    max_episodes=50
)

In [30]:
episodes_by_simon # List of Episode objects

[Episode(title='Best of SingOut SpeakOut No.3', duration_seconds=803.0, podcast_title='SingOut SpeakOut', num_hosts=1, num_guests=0),
 Episode(title='It's All Gone', duration_seconds=360.0, podcast_title='SingOut SpeakOut', num_hosts=1, num_guests=0),
 Episode(title='Today Is Yesterday', duration_seconds=416.0, podcast_title='SingOut SpeakOut', num_hosts=1, num_guests=0),
 Episode(title='Saturn Return', duration_seconds=476.0, podcast_title='SingOut SpeakOut', num_hosts=1, num_guests=0),
 Episode(title='Quarterlife Crisis', duration_seconds=509.0, podcast_title='SingOut SpeakOut', num_hosts=1, num_guests=1)]

#### Working with Conversation Turns

In [31]:
turns = episode.get_all_turns()

print(f"Total turns: {len(turns)}", end="\n\n")

for turn in turns[:5]:
    print(f"Speaker: {turn.primary_speaker}")
    print(f"Duration: {turn.duration:.1f} seconds")
    print(f"Words: {turn.word_count}")
    print(f"Text: {turn.text[:100]}...")
    print("---")

Total turns: 15

Speaker: SPEAKER_00
Duration: 37.2 seconds
Words: 77
Text:  I'm Simon Shapiro and this is Sing Out Speak Out. Here I give you the philosophies, ideas and songs...
---
Speaker: SPEAKER_00
Duration: 0.3 seconds
Words: 1
Text:  Hi...
---
Speaker: SPEAKER_01
Duration: 4.5 seconds
Words: 4
Text:  everyone, welcome to another...
---
Speaker: SPEAKER_01
Duration: 1.6 seconds
Words: 1
Text:  episode...
---
Speaker: SPEAKER_00
Duration: 179.6 seconds
Words: 595
Text:  of Sing Out Speak Out. Exciting news today, we have just released, I want to say we, I mean a coupl...
---


In [32]:
# Calculate basic turn statistics
durations = [turn.duration for turn in turns]
word_counts = [turn.word_count for turn in turns]

print(f"Average turn duration: {sum(durations) / len(durations):.1f} seconds")
print(f"Average words per turn: {sum(word_counts) / len(word_counts):.1f}")
print(f"Longest turn: {max(durations):.1f} seconds")
print(f"Shortest turn: {min(durations):.1f} seconds")
print(f"Total conversation time: {sum(durations) / 60:.1f} minutes")

Average turn duration: 34.0 seconds
Average words per turn: 47.0
Longest turn: 179.6 seconds
Shortest turn: 0.3 seconds
Total conversation time: 8.5 minutes


In [33]:
# Get turns from first 5 minutes
early_turns = episode.get_turns_by_time_range(0, 300)

# Get turns from last 10 minutes
late_turns = episode.get_turns_by_time_range(
    episode.duration_seconds - 600,
    episode.duration_seconds
)

# Get all turns by a specific speaker
speaker_turns = episode.get_turns_by_speaker("SPEAKER_00")

# Get all turns by host
host_turns = episode.get_host_turns()

# Get all turns by guests
guest_turns = episode.get_guest_turns()

# Get turns with at least 50 words
long_turns = episode.get_turns_by_min_length(50)

### Problems with data
The dataset is very detailed and straightforward to work with using the sporc library, but there are still some issues with the data to be ware of. We have found the following issues, which we will adress and think about throughout the analysis to be performed in the project:
- Inaccuracies in the labeled speaker turns
    - It is observed that one actual speaker turn is sometimes divided into smaller turns following each other by the same speaker, without the turn actually being broken
    - It is also observed that podcasts without guests and only one host sometimes have turns labeled with speaker 0 and speaker 1 despite there only being one speaker
    - Some turns metadata is also sometimes wrong, e.g. inaccurate turn duration
    - Speaker labels are inferred using NER and heuristics, so they may be misassigned
- Some episodes don't have any turns
    - Despite a transscript of an episode existing there may be no turns associated with that episode, which makes some sorts of analysis impossible

There might be more issues, which we have not uncovered in this initial inspection of data, but it may uncover itself during the analysis performed until P3 milestone. All of the above issues will be thought out carefully in the relation to the analysis, and will be handled and reported accordingly to be transparant about possible effects e.g. for bad turn labeling, metadata and splitting for interruption analysis, where this information is key for detection of interruptions through overlapping turns by different speakers through e.g. start and end times.

## Approach to sentiment analysis
1. Use existing lexicons of sentiments e.g. VADER, nrc, AFINN, Bing (combination of emotions and categorical/numeric scores)
2. HuggingFace fine-tuned transformer 
3. Other approaches?

Try to run the different methods on the data and do proxy evaluations with no ground truth by:
- Comparing inter-method agreement on sentiments and polarity
- Do a sanity check of a sample
- Pull samples with highest and lowest confidence and check for "traps" giving wrong labels or "shortcuts" to the right label
- Check which words are often found togehter with negations like not, cant etc., to see if that could influence the results + do an actual flip test by testing statements and the same statement prefixed with not to compare sentiments and scores (robustness)

When choosing a method also run an evaluation on a known emotion/sentiment benchmark to make sure performance there is also reasonable as an additonal sanity check.

When landing on an approach the analysis should consist of sentiment analysis and descriptive overviews and comparisons as well as time series overview of sentiment over an episode length. Both should be supported by nice visualizations to tell the story of the use of emotions and polarity in the podcasts across each genre. Inspired to some degree by this report: https://www.kaggle.com/code/josephnehrenz/nlp-sentiment-analysis-of-joe-rogan-experience/report but with added elements. 

## Approach to interruptions analysis
Bla bla - Mikkel skriv her

In [ ]:
# initial idea for detection of interruptions
episodes = sporc.get_all_episodes()

for episode in episodes:
    turns = episode.get_all_turns()
    for i in range(len(turns)-1):
        if turns[i].end_time > turns[i+1].start_time:
            print(f"{episode.title}: {turns[i].speaker} interrupted {turns[i+1].speaker} at {turns[i].end_time:.2f}s (overlap {turns[i].end_time - turns[i+1].start_time:.2f}s)")

# Note: We need to consider consequitive turns by the same speaker, which should not count as an interruption by oneself         

## Approach to Call-to-Action (CTA) Detection and Analysis

#### 1. Detection Methods to consider:
- Rule-based detection using NLTK for imperative verbs and command structures
- LLM-based detection using zero/few-shot prompting
- Hybrid approach combining both methods

#### 2. Text Processing Approaches:
- Turn-level analysis: Analyze each conversation turn independently
    - Pros: Natural conversation boundaries, speaker context
    - Cons: Some episodes lack turn annotations, turns can be imperfect
- Sliding window analysis: Process transcript with overlapping chunks
    - Pros: Handles episodes without turns, maintains context
    - Cons: May split CTAs across chunks, loses speaker information

#### 3. Possible Evaluation Strategy:
- Manual annotation of sample data for benchmarking
- Compare agreement between rule-based and LLM approaches
- Analyze false positives/negatives to refine detection
- Test robustness across different podcast genres

#### 4. Analysis Plan:
- CTA frequency across genres and episode types
- Temporal distribution within episodes
- Speaker analysis (host vs guest CTAs)
- Common CTA types and patterns